In [ ]:
#!/usr/bin/env python
# coding: utf-8
import torch
from diffusers import StableDiffusionXLPipeline, DDIMScheduler, AutoencoderKL
from PIL import Image

from ip_adapter import IPAdapterPlusXL
from ip_adapter.custom_pipelines import StableDiffusionXLCustomPipeline
import wandb
from ip_adapter.utils import register_cross_attention_hook, get_net_attn_map, attnmaps2images

In [ ]:
base_model_path = "stabilityai/stable-diffusion-xl-base-1.0"
image_encoder_path = "models/image_encoder/"
vae_model_path = "stabilityai/sd-vae-ft-mse"
ip_ckpt = "sdxl_models/ip-adapter-plus-face_sdxl_vit-h.bin" # a experimental version
device = "cuda"

In [ ]:
pipe = StableDiffusionXLCustomPipeline.from_pretrained(
    base_model_path,
    torch_dtype=torch.float16,
    add_watermarker=False,
)
pipe.unet = register_cross_attention_hook(pipe.unet)

In [ ]:
pipe.unet = register_cross_attention_hook(pipe.unet)

In [ ]:
ip_model = IPAdapterPlusXL(pipe, image_encoder_path, ip_ckpt, device, num_tokens=16)
# load ip-adapter

In [ ]:
face_image = Image.open("assets/images/ai_face.png")
# generate image
prompt = "photo of a beautiful girl wearing casual shirt in a garden"
negative_prompt = "monochrome, lowres, bad anatomy, worst quality, low quality, blurry"
face_image.resize((256, 256))


In [ ]:
images = ip_model.generate(
        pil_image=face_image,
        prompt=prompt,
        num_samples=1,
        num_inference_steps=30,
        width=512, height=768,
        seed=420)
images[0]

In [ ]:
attn_maps = get_net_attn_map((768, 512))
print(attn_maps.shape)
attn_hot = attnmaps2images(attn_maps)

In [ ]:
import matplotlib.pyplot as plt
#axes[0].imshow(attn_hot[0], cmap='gray')
display_images = [face_image] + attn_hot + [images[0]]
fig, axes = plt.subplots(1, len(display_images), figsize=(12, 4))
for axe, image in zip(axes, display_images):
    axe.imshow(image, cmap='gray')
    axe.axis('off')
plt.show()